# NoSQL - TP2

> **Avant de l'exécuter**
> - Placez-le dans le dossier `TP2/`, à côté du Secure Connect Bundle, de `stations.csv` et de `status.csv`.
> - Le fichier `stations.csv` doit contenir `station_id, station_code, name, commune, lat, lon, capacity, rental_methods` ; `status.csv` doit contenir `station_id, horodatage, mechanical, electrical, num_docks_available, is_renting` (et éventuellement `day`).

# 0. Mise en place

In [ ]:
!pip install "numpy<2" pandas cassandra-driver --quiet

In [ ]:
# pip install cassandra-driver python-dotenv pandas
import os
import re
from datetime import datetime, timedelta
from string import Template

import pandas as pd
from cassandra.auth import PlainTextAuthProvider
from cassandra.cluster import Cluster
from cassandra.concurrent import execute_concurrent_with_args
from cassandra.query import UNSET_VALUE, BatchStatement

# Remplacer le token par celui de votre instance Astra CS. Le Secure Connect Bundle doit être dans le même dossier que ce notebook.
cluster = Cluster(
    cloud={"secure_connect_bundle": "secure-connect-but-sd.zip"},
    auth_provider=PlainTextAuthProvider("token", "AstraCS:..."),
)
session = cluster.connect("velib")
print("Cassandra", session.execute("SELECT release_version FROM system.local").one().release_version)

In [ ]:
# Les méthodes définies ici sont des utilitaires pour exécuter des requêtes et afficher les résultats. Elles ne font pas partie de Cassandra.
pd.set_option("display.max_columns", None)


def run(cql, expect_error=False):
    """Exécute une requête CQL et retourne le résultat sous forme de DataFrame.
    Avec expect_error=True, affiche l'erreur de Cassandra au lieu de la lever."""
    print(cql.strip(), end="\n\n")
    try:
        df = pd.DataFrame(list(session.execute(cql)))
    except Exception as e:
        if not expect_error:
            raise
        print(f"❌ {type(e).__name__} : {e}")
        return
    if df.empty:
        print("(pas de ligne à afficher)")
        return
    return df


def describe(table):
    """Équivalent de DESCRIBE TABLE dans cqlsh."""
    print(cluster.metadata.keyspaces["velib"].tables[table].export_as_string())


def load(cql, rows, concurrency=50):
    """Insère une liste de tuples avec une requête préparée, en parallèle."""
    prepared = session.prepare(cql)
    execute_concurrent_with_args(session, prepared, rows, concurrency=concurrency)
    print(f"{len(rows)} lignes insérées")

In [ ]:
stations = pd.read_csv("stations.csv", dtype={"station_code": str})
status = pd.read_csv("status.csv")
stations["commune"] = stations["commune"].fillna("")

print(len(stations), "stations ;", len(status), "mesures")


def to_set(value):
    """'CREDITCARD|KEY' ou "['CREDITCARD', 'KEY']" -> {'CREDITCARD', 'KEY'} ; valeur absente -> None."""
    found = set(re.findall(r"[A-Z_]+", str(value))) if pd.notna(value) else set()
    return found or None


ts = pd.to_datetime(status["ts"], utc=True)
status["ts"] = ts.dt.tz_localize(None)        # UTC « naïf », comme l'attend le pilote
status["day"] = ts.dt.date                              # la partition est le jour UTC
status["is_renting"] = status["is_renting"].astype(str).str.lower().isin(["true", "1"])

display(stations.head(3))
display(status.head(3))

# 2. Le modèle Cassandra

Test du principe avec la table d'exemple `station_demo`.

In [ ]:
run("""
CREATE TABLE IF NOT EXISTS station_demo (
    station_id int,
    horodatage timestamp,
    name       text,
    capacity   int,
    PRIMARY KEY (station_id, horodatage)
) WITH CLUSTERING ORDER BY (horodatage DESC)
""")

run("INSERT INTO station_demo (station_id, horodatage, name, capacity) VALUES (427, '2026-10-04 09:00:00+0000', 'Château de Vincennes', 46)")
run("INSERT INTO station_demo (station_id, horodatage, name, capacity) VALUES (427, '2026-10-04 09:05:00+0000', 'Château de Vincennes', 46)")
run("INSERT INTO station_demo (station_id, horodatage, name, capacity) VALUES (501, '2026-10-04 09:00:00+0000', 'Mairie de Vincennes', 30)")

In [ ]:
run("SELECT * FROM station_demo WHERE station_id = 427")

In [ ]:
run("""
SELECT * FROM station_demo
WHERE station_id = 427
  AND horodatage >= '2026-10-04 09:00:00+0000'
  AND horodatage <  '2026-10-04 09:05:00+0000'
""")

In [ ]:
run("DROP TABLE station_demo")

Le premier `SELECT` rend les deux observations de la station 427, de la plus récente à la plus ancienne (ordre de clustering `DESC`). Le second borne la recherche sur la clé de clustering : seule l'observation de 09:00 est retournée, car la borne supérieure (09:05) est exclue.

# 3. Les informations des stations

**Question 3.1** Créez la table `stations_par_id`, choisissez la clé primaire, affichez sa définition et justifiez votre choix en une phrase.

**Question 3.2** Exécutez le chargement de `stations_par_id` et vérifiez le nombre de lignes.

Le résultat doit être égal au nombre de lignes de `stations.csv`. Comme `INSERT` est un *upsert*, relancer le chargement ne crée pas de doublons.

## 3.3 Écrire les requêtes de lecture

**Questions.**
1. Afficher toutes les informations de la station `427`.
2. Afficher le nom, la commune et la capacité de la station `610`.
3. Afficher uniquement les identifiants et les noms des stations `501`, `502` et `720`, en une seule requête.

Pour chacune, noter les colonnes de la clé primaire utilisées dans le `WHERE`.

## 3.4 Le moment de rupture

**Questions.** Testez, sans `ALLOW FILTERING`, et notez le message de Cassandra :
1. Retrouver les stations de `Nanterre`.
2. Retrouver les stations dont la capacité est strictement supérieure à `30`.
3. Trier toutes les stations par capacité.

**Messages attendus.**
- Requêtes 1 et 2 : `InvalidRequest`, message du type *« Cannot execute this query as it might involve data filtering and thus may have unpredictable performance. If you want to execute this query despite the performance unpredictability, use ALLOW FILTERING »*.
- Requête 3 : `InvalidRequest`, message du type *« ORDER BY is only supported when the partition key is restricted by an EQ or an IN »*.

### Questions

1. **À quoi sert la clé de partition ?**
2. **Pourquoi une requête sans clé de partition peut-elle être coûteuse ?** 
3. **Que faudrait-il changer pour répondre à « stations d'une commune » ?** 

## 3.5 Modifier les données

**Questions.** Écrivez une requête pour chaque modification, puis relisez la ligne :
1. Passer la capacité de la station `501` à `32`.
2. Ajouter `APPLEPAY` à la station `427` sans remplacer les moyens déjà présents.
3. Retirer `KEY` de la station `427`.
4. Modifier le nom de la station `720`.
5. Supprimer la colonne `lon` de la station `611` sans supprimer le reste de la station.

### Réponse

# 4. Historiser les mesures

## 4.1 Définir les besoins de lecture

**Question.** Créez `station_historique_journalier` : une partition par station et par jour, mesures triées par horodatage décroissant, durée de vie par défaut de 30 jours. Affichez la définition et expliquez la clé de partition et la clé de clustering.

## 4.2 Charger les mesures

## 4.3 Écrire des requêtes temporelles

**Questions.** Pour la station `427` :
1. Afficher les quatre dernières mesures du `4 octobre 2026`.
2. Afficher uniquement l'horodatage, le nombre de vélos mécaniques et le nombre de bornes libres pour cette journée, limités à 10 lignes.
3. Afficher les mesures comprises entre `09:00` et `09:30`, inclusivement.
4. Afficher les mesures de cette même plage dans l'ordre chronologique.
5. Afficher les mesures de la station `501` pour cette même journée.
6. Insérer une mesure pour `427` avec `is_renting = false` à un horodatage où une mesure existe déjà. Que se passe-t-il ?


In [ ]:
# 1. L'ordre de clustering est DESC : les dernières mesures sont les premières lignes


In [ ]:
# 2. Projection de trois colonnes


In [ ]:
# 3. Intervalle sur la clé de clustering (bornes incluses)


In [ ]:
# 4. ORDER BY ASC : l'inverse de l'ordre déclaré, autorisé car la partition est fixée


In [ ]:
# 5. Autre station : autre partition, pas de mélange


In [ ]:
# 6. Réécriture d'une mesure existante : on prend l'horodatage de la première mesure de la partition


### Requête sans station ni jour

**Question.** Testez une requête qui ne donne que l'horodatage. Relevez l'erreur et expliquez pourquoi la clé de partition doit être fournie.

## 4.4 Comprendre le découpage par jour

**Questions.**
1. Écrire les requêtes pour les mesures du `3` et du `4 octobre` de la station `427`. Peut-on le faire en une seule requête ?
2. Comment une application déterminerait-elle les jours concernés par une période de 24 heures et combinerait-elle les résultats ?
3. Calculer le nombre maximal de lignes par partition avec une mesure par minute. Quel est l'intérêt de cette borne ?
4. Quel est le compromis entre un bucket par jour et un bucket par heure ?

In [ ]:
# 1. Une requête par jour...

In [ ]:
# ... ou une seule requête avec IN sur la partie « day » de la clé de partition


1. On peut utiliser `IN` sur `day` : c'est autorisé, car `station_id` est fixé. Mais le coordinateur interroge quand même **deux partitions**, et les résultats sont regroupés par partition, pas triés globalement par date. Pour de nombreux jours, il vaut mieux lancer une requête par jour **en parallèle** et fusionner côté application.
2. L'application calcule les jours UTC couverts par `[début, fin]`, lance une requête par jour (avec bornes d'horodatage), puis concatène et trie les résultats. Une période de 24 heures touche au plus **deux** jours.

In [ ]:
ps_hist = session.prepare(
    "SELECT * FROM station_historique_journalier "
    "WHERE station_id = ? AND day = ? AND horodatage >= ? AND horodatage <= ?"
)


def history(station_id, start, end):
    """Mesures de la station entre start et end (datetime UTC naïfs), toutes journées confondues."""
    days = pd.date_range(start.date(), end.date(), freq="D").date
    futures = [session.execute_async(ps_hist, (station_id, d, start, end)) for d in days]   # une requête par partition
    rows = [r for f in futures for r in f.result()]
    return pd.DataFrame(rows).sort_values("horodatage") if rows else pd.DataFrame()


end = status["horodatage"].max().to_pydatetime()
start = end - timedelta(hours=24)
result = history(P["st1"], start, end)
print(f"{len(result)} mesures entre {start} et {end}")
result.head()

In [ ]:
# 3. Borne théorique avec une mesure par minute, et valeur observée


# 5. Classer les stations d'une commune

## 5.1 Besoin de l'application

**Question.** Dessinez la clé primaire de `stations_par_commune`, identifiez la clé de partition et les colonnes de clustering, puis créez la table.

## 5.2 Charger les données

**Question.** Calculez, avec pandas, le nombre de vélos disponibles de la dernière mesure de chaque station, puis chargez la table.

La même station est maintenant écrite dans `stations_par_id` (informations) et dans `stations_par_commune` (classement) : c'est la **dénormalisation**.

## 5.3 Écrire les requêtes

**Questions.**
1. Afficher les stations de `Nanterre` dans l'ordre défini par la table.
2. Afficher les deux premières.
3. Afficher les stations de `Saint-Mandé`.
4. Afficher les stations de `Nanterre` sans aucun vélo (ajouter une station appropriée si besoin).
5. Retrouver la station `502` dans la partition `Nanterre`.

In [ ]:
# 4. On ajoute d'abord une station de test sans vélo, puis on la recherche (égalité sur la 1re colonne de clustering)


In [ ]:
# 5. station_id est la 2e colonne de clustering : on ne peut pas la restreindre seule


1. Une seule partition, lue dans l'ordre de stockage : du plus grand nombre de vélos au plus petit.
2. `LIMIT 2` donne les deux stations les mieux fournies (les premières lignes de la partition).
3. Même requête sur une autre partition.
4. `num_bikes_available = 0` est une égalité sur la **première** colonne de clustering : autorisé, lecture directe.
5. **Refusé.** Message du type *« PRIMARY KEY column "station_id" cannot be restricted as preceding column "num_bikes_available" is not restricted »*. On ne peut restreindre une colonne de clustering que si les précédentes le sont. Les solutions sont ci-dessous.

In [ ]:
# 5. Solution A : fournir aussi le nombre de vélos (connu ici par la dernière mesure)


In [ ]:
# 5. Solution B : ALLOW FILTERING, acceptable ici car le filtrage reste dans UNE partition


## 5.4 Mettre à jour une clé de clustering

**Questions.** La station `427` passe à 3 vélos disponibles.
1. Essayez un `UPDATE` direct de `num_bikes_available`.
2. Relevez et expliquez le résultat.
3. Écrivez les opérations pour supprimer l'ancienne ligne et insérer la nouvelle.
4. Relisez la partition et vérifiez que l'ancienne disponibilité a disparu.

In [ ]:
row = ranking[ranking["station_id"] == P["st1"]].iloc[0]
commune, old_bikes, name = row["commune"], int(row["num_bikes_available"]), row["name"]
print(f"Station {P['st1']} : {name} ({commune}), {old_bikes} vélos avant la modification")

# 1. UPDATE direct d'une colonne de la clé primaire


2. 

In [ ]:
# 3. Suppression + insertion dans un même lot : même partition, donc atomique et isolé


In [ ]:
# 4. Relecture de la partition


# 6. Concevoir une table à partir d'un besoin

**Question.** La ville veut les stations d'une commune ouvertes à la location, sans parcourir toutes les stations.


4. **Chargement** : à partir de la dernière mesure de chaque station (`is_renting`).

5. **Requêtes** : stations ouvertes, puis fermées, d'une même commune (on choisit une commune qui a les deux états).

In [ ]:
# 6. Toutes les stations ouvertes, sans préciser de commune
